In [ ]:
!pip install transformers datasets evaluate rouge_score nltk

In [ ]:
import torch
import numpy as np
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
dataset = load_dataset("knkarthick/samsum")

print(dataset)

In [ ]:
model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

In [ ]:
def preprocess_function(examples):

    inputs = tokenizer(
        examples["dialogue"],
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=64,
        truncation=True
    )

    inputs["labels"] = labels["input_ids"]

    return inputs

In [ ]:
dataset_tokenized = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
rouge = evaluate.load("rouge")

In [ ]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return result

In [ ]:
training_args = TrainingArguments(
    output_dir="samsum-model",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    eval_accumulation_steps=8
)

In [ ]:
small_train = dataset_tokenized["train"].select(range(300))
small_eval = dataset_tokenized["validation"].select(range(100))

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

predictions = []
references = []

for i in range(50):   # evaluate only 50 samples

    dialogue = dataset["validation"][i]["dialogue"]
    reference = dataset["validation"][i]["summary"]

    inputs = tokenizer(dialogue, return_tensors="pt", truncation=True).to(device)

    output = model.generate(inputs["input_ids"], max_length=60)

    pred = tokenizer.decode(output[0], skip_special_tokens=True)

    predictions.append(pred)
    references.append(reference)

result = rouge.compute(predictions=predictions, references=references)

print(result)

In [ ]:
text = """
Amanda: Are we meeting today?
Jerry: Yes at 6 pm.
Amanda: Perfect see you there.
"""

iinputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

In [ ]:
text = dataset["validation"][0]["dialogue"]

inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=5,
    length_penalty=2.0,
    early_stopping=True
)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

In [ ]:
!jupyter nbconvert --to notebook --ClearOutputPreprocessor.enabled=True text_summeriser_using_lm.ipynb